<a href="https://colab.research.google.com/github/p-tech/wbs-dm-2026/blob/main/web_scrape/RightMove_Extract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [85]:
!pip install requests beautifulsoup4 pandas

In [108]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time


In [120]:
# The base URL for the search
BASE_SEARCH_URL = "https://www.rightmove.co.uk/property-for-sale/find/Walmsleys-The-Way-to-Move/Coventry.html?locationIdentifier=BRANCH%5E248888&includeSSTC=true&_includeSSTC=on"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36",
    "Accept-Language": "en-GB,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Referer": "[https://www.google.com/](https://www.google.com/)"
}

In [121]:
def get_page_soup(url):
    session = requests.Session()
    try:
        response = session.get(url, headers=HEADERS, timeout=10)
        response.raise_for_status() # Check for 403 or 404 errors
        return BeautifulSoup(response.text, "html.parser")
    except requests.exceptions.HTTPError as e:
        print(f"Error: {e}")
        return None

soup = get_page_soup(TARGET_URL)

In [132]:
def extract_all_rightmove_properties(base_url, max_pages=3):
    all_links = []

    for page in range(max_pages):
        # Rightmove pagination: index=0, index=24, index=48, etc.
        index = page * 24
        url = f"{base_url}&index={index}"
        print(f"Scraping page {page + 1}: {url}")

        soup = get_page_soup(url)
        if not soup:
            break

        # Step 1: Find all property cards using the specific class
        property_cards = soup.find_all("div", class_="PropertyCard_propertyCardContainer__VSRSA")

        page_links_count = 0
        for card in property_cards:
            # Step 2: Find the link within the card
            # Note: The class name might be on the 'a' tag directly or an internal element
            link_tag = card.find("a", href=True)

            if link_tag:
                href = link_tag.get("href")

                # Step 3: Ensure the URL is absolute
                if href.startswith("/"):
                    href = "https://www.rightmove.co.uk" + href

                # Check for property details pattern and avoid duplicates
                if "/properties/" in href and href not in all_links:
                    all_links.append(href)
                    page_links_count += 1

        print(f"Found {page_links_count} new links on this page.")

        # Ethical scraping: respect the server with a short delay
        time.sleep(2)

    return all_links

# Run the extraction (limited to 3 pages for this exercise)
property_urls = extract_all_rightmove_properties(BASE_SEARCH_URL, max_pages=1)

print(f"\nTotal unique property URLs extracted: {len(property_urls)}")
for i, url in enumerate(property_urls[:5]):
    print(f"{i+1}: {url}")


Scraping page 1: https://www.rightmove.co.uk/property-for-sale/find/Walmsleys-The-Way-to-Move/Coventry.html?locationIdentifier=BRANCH%5E248888&includeSSTC=true&_includeSSTC=on&index=0
Found 25 new links on this page.

Total unique property URLs extracted: 25
1: https://www.rightmove.co.uk/properties/164598299#/?channel=RES_BUY
2: https://www.rightmove.co.uk/properties/168245339#/?channel=RES_BUY
3: https://www.rightmove.co.uk/properties/157014713#/?channel=RES_BUY
4: https://www.rightmove.co.uk/properties/164441120#/?channel=RES_BUY
5: https://www.rightmove.co.uk/properties/168250622#/?channel=RES_BUY


In [133]:
df = pd.DataFrame(property_urls, columns=["Property_URL"])

df.head()

,Property_URL
0,https://www.rightmove.co.uk/properties/1645982...
1,https://www.rightmove.co.uk/properties/1682453...
2,https://www.rightmove.co.uk/properties/1570147...
3,https://www.rightmove.co.uk/properties/1644411...
4,https://www.rightmove.co.uk/properties/1682506...


In [136]:
def scrape_property_details(urls):
    details = []

    for i, url in enumerate(urls):
        print(f"[{i+1}/{len(urls)}] Scraping details for: {url}")

        soup = get_page_soup(url)
        if soup:
            # Extract the H1 tag element (Property Title/Address)
            h1_tag = soup.find("h1")
            title = h1_tag.get_text(strip=True) if h1_tag else "N/A"

            # Extract the Price using the specific class
            price_tag = soup.find("div", class_="_1gfnqJ3Vtd1z40MlC0MzXu")
            price = price_tag.get_text(strip=True) if price_tag else "N/A"

            details.append({
                "Property_URL": url,
                "Property_Title": title,
                "Price": price
            })

        else:
            details.append({
                "Property_URL": url,
                "Property_Title": "Failed to load",
                "Price": "Failed to load"
            })

        # Short delay to prevent being blocked during deep scraping
        time.sleep(1.5)

    return details

# Execute the detail extraction
property_data = scrape_property_details(property_urls)

[1/25] Scraping details for: https://www.rightmove.co.uk/properties/164598299#/?channel=RES_BUY
[2/25] Scraping details for: https://www.rightmove.co.uk/properties/168245339#/?channel=RES_BUY
[3/25] Scraping details for: https://www.rightmove.co.uk/properties/157014713#/?channel=RES_BUY
[4/25] Scraping details for: https://www.rightmove.co.uk/properties/164441120#/?channel=RES_BUY
[5/25] Scraping details for: https://www.rightmove.co.uk/properties/168250622#/?channel=RES_BUY
[6/25] Scraping details for: https://www.rightmove.co.uk/properties/162876110#/?channel=RES_BUY
[7/25] Scraping details for: https://www.rightmove.co.uk/properties/161480897#/?channel=RES_BUY
[8/25] Scraping details for: https://www.rightmove.co.uk/properties/166642319#/?channel=RES_BUY
[9/25] Scraping details for: https://www.rightmove.co.uk/properties/87495255#/?channel=RES_BUY
[10/25] Scraping details for: https://www.rightmove.co.uk/properties/172465706#/?channel=RES_BUY
[11/25] Scraping details for: https://ww

In [137]:
df = pd.DataFrame(property_data)
df.to_csv("rightmove_detailed_results.csv", index=False)
print(f"Data for {len(df)} properties saved to rightmove_detailed_results.csv")

# Display the first few rows
print(df.head())

Data for 25 properties saved to rightmove_detailed_results.csv
                                        Property_URL  \
0  https://www.rightmove.co.uk/properties/1645982...   
1  https://www.rightmove.co.uk/properties/1682453...   
2  https://www.rightmove.co.uk/properties/1570147...   
3  https://www.rightmove.co.uk/properties/1644411...   
4  https://www.rightmove.co.uk/properties/1682506...   

                              Property_Title  \
0      Shortley Road, Whitley, Coventry, CV3   
1       Morningside, Earlsdon, Coventry, CV5   
2   Armorial Road, Stivichall, Coventry, CV3   
3  Beechwood Avenue, Earlsdon, Coventry, CV5   
4      The Riddings, Earlsdon, Coventry, CV5   

                                               Price  
0  £240,000Knowing the purchase price means you c...  
1  £800,000Knowing the purchase price means you c...  
2  £795,000Knowing the purchase price means you c...  
3  £635,000Knowing the purchase price means you c...  
4  £625,000Knowing the purchase pric